<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


- Install the additional package requirements for this bonus notebook by uncommenting and running the following cell:

In [2]:
! pip3 install -r requirements-extra.txt

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 12.0 MB 7.1 MB/s eta 0:00:01
     |████████████████████████████████| 566 kB 45.4 MB/s eta 0:00:01
     |████████████████████████████████| 3.0 MB 66.4 MB/s eta 0:00:01
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 1.8.0
    Uninstalling huggingface-hub-1.8.0:
      Successfully uninstalled huggingface-hub-1.8.0
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


# Comparing Various Byte Pair Encoding (BPE) Implementations

<br>
&nbsp;

## 1. Using BPE from `tiktoken`

In [3]:
from importlib.metadata import version

print("tiktoken version:", version("tiktoken"))

tiktoken version: 0.13.0


In [4]:
import tiktoken

tik_tokenizer = tiktoken.get_encoding("gpt2")

text = "Hello, world. Is this-- a test?"

In [5]:
integers = tik_tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

[15496, 11, 995, 13, 1148, 428, 438, 257, 1332, 30]


In [6]:
strings = tik_tokenizer.decode(integers)

print(strings)

Hello, world. Is this-- a test?


In [7]:
print(tik_tokenizer.n_vocab)

50257


<br>
&nbsp;

## 2. Using the original BPE implementation used in GPT-2

In [8]:
from bpe_openai_gpt2 import get_encoder, download_vocab

/Users/sunghoojung/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [9]:
download_vocab()

Fetching encoder.json: 1.04Mit [00:00, 2.95Mit/s]                                                   
Fetching vocab.bpe: 457kit [00:00, 1.86Mit/s]                                                       


In [10]:
orig_tokenizer = get_encoder(model_name="gpt2_model", models_dir=".")

In [11]:
integers = orig_tokenizer.encode(text)

print(integers)

[15496, 11, 995, 13, 1148, 428, 438, 257, 1332, 30]


In [12]:
strings = orig_tokenizer.decode(integers)

print(strings)

Hello, world. Is this-- a test?


<br>
&nbsp;

## 3. Using the BPE via Hugging Face transformers

In [13]:
import transformers

transformers.__version__

'4.57.6'

In [14]:
from transformers import GPT2Tokenizer

hf_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

In [15]:
hf_tokenizer(strings)["input_ids"]

[15496, 11, 995, 13, 1148, 428, 438, 257, 1332, 30]

In [16]:
from transformers import GPT2TokenizerFast

hf_tokenizer_fast = GPT2TokenizerFast.from_pretrained("gpt2")

In [17]:
hf_tokenizer_fast(strings)["input_ids"]

[15496, 11, 995, 13, 1148, 428, 438, 257, 1332, 30]

<br>
&nbsp;

## 4. Using my own from-scratch BPE tokenizer

In [18]:
import os
import sys
import io
import nbformat
import types

def import_from_notebook():
    def import_definitions_from_notebook(fullname, names):
        current_dir = os.getcwd()
        path = os.path.join(current_dir, "..", "05_bpe-from-scratch", fullname + ".ipynb")
        path = os.path.normpath(path)

        # Load the notebook
        if not os.path.exists(path):
            raise FileNotFoundError(f"Notebook file not found at: {path}")

        with io.open(path, "r", encoding="utf-8") as f:
            nb = nbformat.read(f, as_version=4)

        # Create a module to store the imported functions and classes
        mod = types.ModuleType(fullname)
        sys.modules[fullname] = mod

        # Go through the notebook cells and only execute function or class definitions
        for cell in nb.cells:
            if cell.cell_type == "code":
                cell_code = cell.source
                for name in names:
                    # Check for function or class definitions
                    if f"def {name}" in cell_code or f"class {name}" in cell_code:
                        exec(cell_code, mod.__dict__)
        return mod

    fullname = "bpe-from-scratch"
    names = ["BPETokenizerSimple"]

    return import_definitions_from_notebook(fullname, names)

In [19]:
imported_module = import_from_notebook()
BPETokenizerSimple = getattr(imported_module, "BPETokenizerSimple", None)

tokenizer_gpt2 = BPETokenizerSimple()
tokenizer_gpt2.load_vocab_and_merges_from_openai(
    vocab_path=os.path.join("gpt2_model", "encoder.json"),
    bpe_merges_path=os.path.join("gpt2_model", "vocab.bpe")
)

In [20]:
integers = tokenizer_gpt2.encode(text)

print(integers)

[15496, 11, 995, 13, 1148, 428, 438, 257, 1332, 30]


<br>
&nbsp;

## 5. A quick performance benchmark

In [21]:
with open("../01_main-chapter-code/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

&nbsp;
### 5.1 Original OpenAI GPT-2 tokenizer

In [22]:
%timeit orig_tokenizer.encode(raw_text)

4.51 ms ± 770 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


&nbsp;
### 5.2 Tiktoken OpenAI GPT-2 tokenizer

In [23]:
%timeit tik_tokenizer.encode(raw_text)

923 µs ± 5.83 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


&nbsp;
### 5.3 Hugging Face OpenAI GPT-2 tokenizer

In [24]:
%timeit hf_tokenizer(raw_text)["input_ids"]

Token indices sequence length is longer than the specified maximum sequence length for this model (5145 > 1024). Running this sequence through the model will result in indexing errors


11.7 ms ± 390 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [25]:
%timeit hf_tokenizer(raw_text, max_length=5145, truncation=True)["input_ids"]

11.3 ms ± 129 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [26]:
%timeit hf_tokenizer_fast(raw_text)["input_ids"]

Token indices sequence length is longer than the specified maximum sequence length for this model (5145 > 1024). Running this sequence through the model will result in indexing errors


3.65 ms ± 74.2 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [27]:
%timeit hf_tokenizer_fast(raw_text, max_length=5145, truncation=True)["input_ids"]

3.61 ms ± 72.9 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


&nbsp;
### 5.4 My own GPT-2 tokenizer (for educational purposes)

In [28]:
%timeit tokenizer_gpt2.encode(raw_text)

15.8 ms ± 355 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
